# Target Services Workflow

This notebook demonstrates the explicit ``PinchProblem.target_*`` workflow on
a packaged Chocolate Factory sample. The packaged
``chocolate_factory.json`` asset is derived from
``examples/OpenPinchWkbs/UnderReview/Chocolote Factory.xlsb`` and is the
supported notebook input for this workflow.


In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

from OpenPinch import PinchProblem
from OpenPinch.lib.enums import HPRcycle
from OpenPinch.resources import copy_sample_case

workspace = TemporaryDirectory()
work_dir = Path(workspace.name)
case_path = copy_sample_case(
    "chocolate_factory.json",
    work_dir / "chocolate_factory.json",
)
problem = PinchProblem(problem_filepath=case_path)
validated = problem.validate()
case_path.name, len(validated.streams), len(validated.utilities)


('chocolate_factory.json', 65, 5)

## Baseline solve

Keep ``target()`` as the baseline full-problem solve. The advanced services are
explicit follow-on entrypoints that operate on the prepared zone tree and refresh
``problem.results`` in place.


In [ ]:
baseline = problem.target()
summary = problem.summary_frame()
summary[
    summary["Target"].isin(
        [
            "chocolate_factory/Direct Integration",
            "Vacuum/Direct Integration",
            "chocolate_factory/Total Site Target",
        ]
    )
].reset_index(drop=True)


## Apply advanced target services

Use a stable public option set for the HPR solves so the notebook remains a
known-good workflow in automated smoke tests. Direct targets are applied to the
``Vacuum`` subzone, while indirect targets are applied at the root site level.


In [ ]:
target_options = {
    "MAX_HP_MULTISTART": 1,
    "HPR_TYPE": HPRcycle.MultiTempCarnot.value,
}

direct_hp = problem.target_direct_heat_pump(
    zone_name="Vacuum",
    options=target_options,
)
direct_r = problem.target_direct_refrigeration(
    zone_name="Vacuum",
    options=target_options,
)
indirect_hp = problem.target_indirect_heat_pump(options=target_options)
indirect_r = problem.target_indirect_refrigeration(options=target_options)
cogeneration = problem.target_cogeneration(zone_name="Vacuum")
area_cost = problem.target_area_cost(zone_name="Vacuum")

pd.DataFrame(
    [
        {
            "target": direct_hp.name,
            "type": direct_hp.type,
            "utility_total": direct_hp.hpr_utility_total,
            "work": direct_hp.hpr_work,
            "external_utility": direct_hp.hpr_external_utility,
        },
        {
            "target": direct_r.name,
            "type": direct_r.type,
            "utility_total": direct_r.hpr_utility_total,
            "work": direct_r.hpr_work,
            "external_utility": direct_r.hpr_external_utility,
        },
        {
            "target": indirect_hp.name,
            "type": indirect_hp.type,
            "utility_total": indirect_hp.hpr_utility_total,
            "work": indirect_hp.hpr_work,
            "external_utility": indirect_hp.hpr_external_utility,
        },
        {
            "target": indirect_r.name,
            "type": indirect_r.type,
            "utility_total": indirect_r.hpr_utility_total,
            "work": indirect_r.hpr_work,
            "external_utility": indirect_r.hpr_external_utility,
        },
        {
            "target": cogeneration.name,
            "type": cogeneration.type,
            "utility_total": None,
            "work": cogeneration.work_target,
            "external_utility": None,
        },
        {
            "target": area_cost.name,
            "type": area_cost.type,
            "utility_total": None,
            "work": area_cost.area,
            "external_utility": area_cost.total_cost,
        },
    ]
)


## Refreshed public results and HPR stream collections

Each ``target_*`` call refreshes ``problem.results``. The public result models carry
the HPR summary fields and keep ``hpr_hot_streams`` / ``hpr_cold_streams`` as
``StreamCollection`` objects for Python-side inspection.


In [ ]:
refreshed = problem.summary_frame()
refreshed[
    refreshed["Target"].isin(
        [
            direct_hp.name,
            direct_r.name,
            indirect_hp.name,
            indirect_r.name,
            "Vacuum/Direct Integration",
        ]
    )
].reset_index(drop=True)

direct_hp_result = next(
    target for target in problem.results.targets if target.name == direct_hp.name
)
{
    "hpr_cycle": direct_hp_result.hpr_cycle,
    "hot_stream_collection": type(direct_hp_result.hpr_hot_streams).__name__,
    "cold_stream_collection": type(direct_hp_result.hpr_cold_streams).__name__,
    "hot_stream_names": [stream.name for stream in direct_hp_result.hpr_hot_streams],
    "cold_stream_names": [stream.name for stream in direct_hp_result.hpr_cold_streams],
}


## Graphs from the refreshed workflow

The baseline and advanced targets can be graphed through the existing public plotting
helpers. Use the generic ``plot()`` helper when you want a specific graph type such as
the heat-pump or site-utility grand composite curves.


In [ ]:
base_gcc = problem.plot_grand_composite_curve(zone_name="Vacuum/Direct Integration")
hp_gcc = problem.plot(
    zone_name="Direct Heat Pump",
    graph_type="Grand Composite Curve with Heat Pump",
)
site_gcc = problem.plot(
    zone_name="Indirect Heat Pump",
    graph_type="Site Utility Grand Composite Curve",
)
[
    base_gcc.layout.title.text,
    hp_gcc.layout.title.text,
    site_gcc.layout.title.text,
]


## Export the refreshed result set

Because the advanced targets refresh ``problem.results`` in place, the usual export
helpers write the updated workbook and graph assets without any private hooks.


In [ ]:
export_dir = work_dir / "exports"
graph_dir = work_dir / "graphs"
workbook_path = problem.export_excel(export_dir)
graph_paths = problem.export_graphs(
    graph_dir,
    zone_name="Direct Heat Pump",
    graph_type="Grand Composite Curve with Heat Pump",
)
{
    "workbook": workbook_path.name,
    "graphs": [path.name for path in graph_paths],
    "graph_dir_contents": sorted(path.name for path in graph_dir.iterdir()),
}
